# 1. Import Libraries & Setup

In [5]:
import os
import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd().parents[1]
RAW_DIR = BASE_DIR / 'dataset_raw'
DATA_DIR = BASE_DIR / 'data'

# 2. Load Raw GoEmotions Datasets

In [6]:
CSV_FILES = [
    os.path.join(RAW_DIR, "goemotions_1.csv"),
    os.path.join(RAW_DIR, "goemotions_2.csv"),
    os.path.join(RAW_DIR, "goemotions_3.csv"),
]

dfs = [pd.read_csv(file) for file in CSV_FILES]
df_all = pd.concat(dfs, ignore_index=True)

print(f"Total row after merged: {len(df_all):,} rows")

Total row after merged: 211,225 rows


# 3. Core Emotion Extraction

In [7]:
CORE_EMOTIONS = ['joy', 'love', 'surprise', 'anger', 'fear', 'sadness']
COLS_TO_KEEP = ['text'] + CORE_EMOTIONS

df_all = df_all[COLS_TO_KEEP].copy()

# Calculate global frequency of each emotion
emotion_frequency = df_all[CORE_EMOTIONS].sum().sort_values()

print("\n=== Emotion Frequency ===")
print(emotion_frequency)

# Keeping the rarest emotion amongst multiple emotions (if exists)
def get_rare_emotion(row):
    active_emotions = [
        emotion for emotion in CORE_EMOTIONS
        if row[emotion] == 1
    ]

    if not active_emotions:
        return None

    return min(
        active_emotions,
        key=lambda emotion: emotion_frequency[emotion]
    ).capitalize()

df_all['emotion_label'] = df_all.apply(get_rare_emotion, axis=1)

df_filtered = (
    df_all
    .dropna(subset=['emotion_label'])
    [['text', 'emotion_label']]
    .copy()
)

df_filtered.reset_index(drop=True, inplace=True)
print(f"\nRemaining Rows after Extraction: {len(df_filtered):,} rows")


=== Emotion Frequency ===
fear        3197
surprise    5514
sadness     6758
joy         7983
anger       8084
love        8191
dtype: int64

Remaining Rows after Extraction: 38,258 rows


# 4. Export to CSV

In [8]:
os.makedirs(DATA_DIR, exist_ok=True)
OUTPUT_PATH = os.path.join(DATA_DIR, "(prod)6_emotions.csv")

df_filtered.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")
print(f"Data successfully saved to: {OUTPUT_PATH}")

Data successfully saved to: c:\Users\VICTUS\OneDrive\Documents\Edwin's_Project\Moofy\data\(prod)6_emotions.csv
